# 03 - Data Preparation

This notebook converts the raw minute-level electricity readings into a clean hourly dataset for analysis, modeling, and dashboard use.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "DataSet" / "household_power_consumption.csv"
OUTPUT_PATH = PROJECT_ROOT / "outputs" / "prepared_hourly_energy.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

## Load and Standardize Columns

Column names are converted to lowercase snake case so later cells are easier to read.

In [ ]:
raw_df = pd.read_csv(DATA_PATH, na_values=["?"], low_memory=False)
raw_df.columns = [column.strip().lower().replace(" ", "_") for column in raw_df.columns]
display(raw_df.head())

## Convert Date, Time, and Numeric Fields

The original file stores date and time separately. They are combined into a single `datetime` column, and power measurements are converted into numeric types.

In [ ]:
numeric_columns = [
    "global_active_power",
    "global_reactive_power",
    "voltage",
    "global_intensity",
    "sub_metering_1",
    "sub_metering_2",
    "sub_metering_3",
]

clean_df = raw_df.copy()
clean_df["datetime"] = pd.to_datetime(
    clean_df["date"].astype(str) + " " + clean_df["time"].astype(str),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce",
)

for column in numeric_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

clean_df = clean_df.dropna(subset=["datetime"]).sort_values("datetime")
clean_df = clean_df.drop_duplicates(subset=["datetime"]).reset_index(drop=True)
print("Rows after datetime cleaning:", len(clean_df))

## Handle Missing Values

Because the readings are a time series, missing numeric values are interpolated over time. Remaining edge gaps are filled forward and backward.

In [ ]:
clean_df = clean_df.set_index("datetime")
clean_df[numeric_columns] = (
    clean_df[numeric_columns]
    .interpolate(method="time", limit_direction="both")
    .ffill()
    .bfill()
)
clean_df = clean_df.reset_index()
print("Missing values after interpolation:", clean_df[numeric_columns].isna().sum().sum())

## Feature Engineering

The new features support time analysis, appliance-level interpretation, classification, and forecasting.

In [ ]:
clean_df["hour"] = clean_df["datetime"].dt.hour
clean_df["day_of_week"] = clean_df["datetime"].dt.dayofweek
clean_df["month"] = clean_df["datetime"].dt.month
clean_df["year"] = clean_df["datetime"].dt.year
clean_df["is_weekend"] = clean_df["day_of_week"].isin([5, 6]).astype(int)

clean_df["sub_metering_total_wh"] = clean_df[["sub_metering_1", "sub_metering_2", "sub_metering_3"]].sum(axis=1)
clean_df["active_energy_wh"] = clean_df["global_active_power"] * 1000 / 60
clean_df["unmetered_energy_wh"] = (clean_df["active_energy_wh"] - clean_df["sub_metering_total_wh"]).clip(lower=0)

display(clean_df.head())

## Resample to Hourly Data

Hourly data is smaller, easier to visualize, and more suitable for the baseline models used in this project.

In [ ]:
hourly_df = clean_df.set_index("datetime").resample("h").agg({
    "global_active_power": "mean",
    "global_reactive_power": "mean",
    "voltage": "mean",
    "global_intensity": "mean",
    "sub_metering_1": "sum",
    "sub_metering_2": "sum",
    "sub_metering_3": "sum",
    "sub_metering_total_wh": "sum",
    "active_energy_wh": "sum",
    "unmetered_energy_wh": "sum",
}).dropna().reset_index()

hourly_df["hour"] = hourly_df["datetime"].dt.hour
hourly_df["day_of_week"] = hourly_df["datetime"].dt.dayofweek
hourly_df["month"] = hourly_df["datetime"].dt.month
hourly_df["year"] = hourly_df["datetime"].dt.year
hourly_df["is_weekend"] = hourly_df["day_of_week"].isin([5, 6]).astype(int)

threshold = hourly_df["global_active_power"].quantile(0.75)
hourly_df["high_consumption"] = (hourly_df["global_active_power"] >= threshold).astype(int)

hourly_df["lag_1_power"] = hourly_df["global_active_power"].shift(1)
hourly_df["lag_2_power"] = hourly_df["global_active_power"].shift(2)
hourly_df["lag_24_power"] = hourly_df["global_active_power"].shift(24)
hourly_df["rolling_3_power_mean"] = hourly_df["global_active_power"].shift(1).rolling(3, min_periods=1).mean()
hourly_df["rolling_24_power_mean"] = hourly_df["global_active_power"].shift(1).rolling(24, min_periods=1).mean()
hourly_df = hourly_df.bfill().ffill()

print("Hourly shape:", hourly_df.shape)
print("Start:", hourly_df["datetime"].min())
print("End:", hourly_df["datetime"].max())

## Save Prepared Dataset

The prepared file is used by the modeling, evaluation, and dashboard notebooks.

In [ ]:
hourly_df.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)
display(hourly_df.head())